# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**My lane:** Content decline detection for refresh prioritization — helping an editor decide *which* content items to refresh first, using the trailing-90-day performance signals in the starter dataset.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification** (with a ranking use downstream).

The underlying decision is: *"Which content items should an editor prioritize to refresh this week?"* That phrasing sounds like ranking, but the FlyRank data gives me a clean binary outcome to learn from first — `trend_direction == 'down'` (a content item's sessions fell over the last 30 days vs. the previous 30). So the core ML task is binary classification: predict *is this content declining?* from features that are NOT the trend columns themselves. Once I have a predicted probability of decline per item, I can sort by that probability to get the ranked refresh queue an editor actually uses — classification first, ranking as the applied layer on top.

In [1]:
# Confirm the task-type choice against the data: how the label is defined
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('Rows x Cols:', df.shape)
print('\ntrend_direction value counts (this is what the label is built from):')
print(df['trend_direction'].value_counts())


Rows x Cols: (30000, 44)

trend_direction value counts (this is what the label is built from):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining`** = 1 if `trend_direction == 'down'`, else 0.

This is an **observed** outcome, not a hand-defined rule I'm inventing: `trend_direction` is already computed upstream by FlyRank by comparing `sessions_last_30d` against `sessions_prev_30d` — it's a measured before/after change, not a label I'm making up. Per the data skill's leakage warning, `trend_direction` and `trend_pct` are the *source* of this label, so they can never appear as input features — only content-level signals available *before* the outcome window (content type, freshness, word count, historical CTR/position, search-demand columns) are allowed as predictors.

In [2]:
# Build the target and check it's a clean binary split, not a rare-event problem
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
base_rate = df['is_declining'].mean()
print(f'Base rate of decline: {base_rate:.1%}')
df[['content_id', 'trend_direction', 'trend_pct', 'is_declining']].head(5)


Base rate of decline: 54.2%


,content_id,trend_direction,trend_pct,is_declining
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: ROC-AUC, reported against the base rate.**

The base rate of decline is ~54%, so plain accuracy is a weak yardstick — a model that always predicts "declining" already scores ~54%. ROC-AUC measures whether the model ranks genuinely declining items above stable/improving ones *regardless of threshold*, which matches how the output gets used (an editor works down a sorted queue, not a fixed cutoff). I'll also track precision@K (e.g. precision in the top 20% by predicted decline probability), since that's the slice of the queue an editor with limited hours will actually touch.

In [3]:
# Establish the baseline every model has to beat: an AUC of 0.50 (a coin flip / the base rate)
print(f'Baseline to beat -> ROC-AUC: 0.50 (random ranking)')
print(f'Naive "always declining" baseline -> Accuracy: {base_rate:.1%}, Precision: {base_rate:.1%}, Recall: 100%')


Baseline to beat -> ROC-AUC: 0.50 (random ranking)
Naive "always declining" baseline -> Accuracy: 54.2%, Precision: 54.2%, Recall: 100%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item**, summarized over its trailing 90 days, belonging to one client. `content_id` and `client_id` are pseudonyms used only for grouping and for a client-grouped train/test split later — never as model features.

In [4]:
# The unit of analysis: features an editor/model could see BEFORE the outcome window,
# plus the target for reference (never as a feature).
feature_cols = [
    'content_type', 'main_intent', 'word_count', 'char_count',
    'search_volume', 'competition', 'competition_level', 'cpc',
    'content_age_days', 'age_tier', 'days_since_last_update', 'freshness_tier',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
lane_slice = df[['content_id', 'client_id'] + feature_cols + ['is_declining']]
print(f'Unit of analysis: {len(lane_slice):,} rows, one per content item')
lane_slice.head(10)


Unit of analysis: 30,000 rows, one per content item


,content_id,client_id,content_type,main_intent,word_count,char_count,search_volume,competition,competition_level,cpc,content_age_days,age_tier,days_since_last_update,freshness_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,20457.0,10.0,0.67,HIGH,2.05,187,181-365,20,0-30,0.76,10.6,5.88,4.55,0.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,15562.0,90.0,0.01,LOW,0.05,445,365+,25,0-30,0.05,20.3,0.00,10.00,0.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,23643.0,0.0,0.00,LOW,0.00,141,91-180,20,0-30,0.09,36.5,0.00,28.57,0.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,NaN,10.0,0.00,LOW,0.00,463,365+,22,0-30,0.49,6.2,1.28,3.45,0.0,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,17469.0,0.0,0.00,LOW,0.00,263,181-365,14,0-30,0.13,44.0,0.00,24.29,0.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3080.0,18178.0,720.0,1.00,HIGH,1.05,147,91-180,20,0-30,0.03,8.5,0.00,25.00,0.0,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,3059.0,20810.0,0.0,0.00,LOW,0.00,90,31-90,20,0-30,0.00,7.0,0.00,0.00,0.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,NaN,NaN,590.0,0.44,MEDIUM,0.64,445,365+,22,0-30,0.06,21.2,3.57,7.14,0.0,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,3807.0,24228.0,0.0,0.00,LOW,0.00,90,31-90,20,0-30,0.09,46.0,5.88,6.25,0.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,NaN,NaN,0.0,0.00,LOW,0.00,257,181-365,104,91-180,0.16,4.9,0.00,0.00,0.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule (e.g. "flag anything older than 180 days" or "flag anything with CTR below X") fails for three reasons the data itself shows:

1. **No single feature separates the classes cleanly.** Declining items average a lower CTR than non-declining ones, but the two distributions overlap heavily — there's no clean threshold, only a weak, noisy tilt.
2. **Signals interact.** Whether low search volume matters depends on `content_type` and `competition_level` at the same time; a rule would need dozens of nested if/else branches to approximate what a model learns as one weighted combination.
3. **Missingness itself is a signal, not noise.** ~26% of rows are missing `word_count`, and that's not random — it's concentrated entirely in one `content_type` ("keyword article"), while the other two types never have it missing. A hand-written rule using `word_count` would silently misfire on a quarter of the catalog unless someone remembered to special-case it; a model can use a `has_word_count` flag as its own signal instead.

In [5]:
# Evidence for the 'why ML' argument: weak single-feature separation + structured missingness
print('Mean CTR by class (weak, overlapping signal):')
print(df.groupby('is_declining')['ctr'].mean(), '\n')

print('word_count missingness is concentrated by content_type, not random:')
print((df.groupby('content_type')['word_count'].apply(lambda s: s.isna().mean())).mul(100).round(1).astype(str) + '%')


Mean CTR by class (weak, overlapping signal):
is_declining
0    0.731611
1    0.324138
Name: ctr, dtype: float64 

word_count missingness is concentrated by content_type, not random:
content_type
comparison article     0.0%
feedly article         0.0%
keyword article       28.3%
Name: word_count, dtype: str


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — `content_id`/`client_id` are the repo's pseudonyms, not real identifiers
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.